# Epoching + Feature Extraction — EEG OpenBCI

**Pipeline:**
1. Configuración global
2. Carga de canales válidos (del EDA)
3. Filtrado de la señal cruda (bandpass 1–45 Hz + notch 50 Hz)
4. Epoching con ventana de 250 muestras (2 s) y paso de 125 (50 % solape)
5. Rechazo de épocas con amplitud > ±150 µV
6. Extracción de 273 features por época
7. Etiquetado por estímulo / página
8. Guardado del dataset final

---
**Features (273 por época)**

| Bloque | Detalle | N |
|--------|---------|---|
| Potencia absoluta | 5 bandas × 16 canales | 80 |
| Potencia relativa | 5 bandas × 16 canales | 80 |
| Ratios espectrales | 5 ratios × 16 canales | 80 |
| Asimetría hemisférica | 8 pares × 4 bandas (sin delta) | 32 |
| Potencia broadband | total | 1 |
| **Total** | | **273** |

---
## 0 · Configuración — edita aquí antes de correr

In [ ]:
import os

# ── Rutas ────────────────────────────────────────────────────────────────────
DATA_DIR  = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'Datos EEG')
META_CSV  = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'meta_participantes.csv')
OUT_DIR   = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'epochs_output')
os.makedirs(OUT_DIR, exist_ok=True)

# ── Señal ────────────────────────────────────────────────────────────────────
FS            = 125          # Hz
EPOCH_SAMPLES = 250          # 2 s
STEP_SAMPLES  = 125          # 50 % solape
AMP_THRESH    = 150.0        # µV — rechazo de épocas
BANDPASS      = (1.0, 45.0)  # Hz filtro paso-banda
NOTCH_HZ      = 50.0         # Hz filtro notch

# ── Bandas EEG ───────────────────────────────────────────────────────────────
BANDS = {
    'delta': (1.0,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0, 13.0),
    'beta' : (13.0,30.0),
    'gamma': (30.0,45.0),
}

# ── Pares hemisféricos (izquierdo, derecho) ───────────────────────────────────
# Asimetría = ln(P_der) − ln(P_izq); positivo → mayor activación derecha
ASYM_PAIRS = [
    ('Fp1','Fp2'), ('F3','F4'), ('F7','F8'), ('C3','C4'),
    ('T7','T8'),   ('P3','P4'), ('P7','P8'), ('O1','O2'),
]
ASYM_BANDS = ['theta','alpha','beta','gamma']  # 4 bandas × 8 pares = 32

# ── Canales 10-20 (orden OpenBCI Cyton+Daisy 16ch) ──────────────────────────
EEG_COLS  = [f'EXG Channel {i}' for i in range(16)]
CH_NAMES  = ['Fp1','Fp2','C3','C4','P7','P8','O1','O2',
              'F7','F8','F3','F4','T7','T8','P3','P4']
COL_MAP   = dict(zip(CH_NAMES, EEG_COLS))   # ch_name → columna raw

# ── Etiquetado de páginas / estímulos ───────────────────────────────────────
#
# OPCIÓN A — bloque fijo: el experimento tiene páginas de duración conocida.
# Define cuántas páginas y cuántos segundos duró cada una.
# La primera página empieza en el sample 0 (o ajusta PAGE_START_OFFSET_S).
#
# OPCIÓN B — desde archivo CSV con columnas [participant, page, start_sample, end_sample].
# Si tienes ese archivo, pon la ruta en EVENTS_CSV y cambia USE_EVENTS_FILE = True.
#
USE_EVENTS_FILE     = False
EVENTS_CSV          = ''           # ruta al CSV de eventos si USE_EVENTS_FILE = True

PAGE_DURATION_S     = 60           # segundos por página (Opción A)
N_PAGES             = 12           # número de páginas (Opción A)
PAGE_START_OFFSET_S = 0            # segundos iniciales a ignorar (baseline, etc.)

print('Configuración cargada OK')

---
## 1 · Imports y funciones base

In [ ]:
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sp_signal
from scipy.signal import butter, sosfiltfilt, iirnotch

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

print('Imports OK')

---
## 2 · Canales válidos por participante (del EDA)

In [ ]:
# Carga el CSV generado en eda_eeg.ipynb
meta = pd.read_csv(META_CSV)

def valid_channels(participant_id):
    """Retorna lista de ch_name cuyos canales NO estuvieron 100% saturados."""
    row = meta[meta['participant'] == participant_id]
    if row.empty:
        return CH_NAMES  # fallback: todos
    return [ch for ch in CH_NAMES if row[f'sat_{ch}'].values[0] < 100.0]

# Verificación rápida
for pid in meta['participant'].head(5):
    valid = valid_channels(pid)
    print(f'{pid}: {len(valid)}/16 canales válidos  → {valid}')

---
## 3 · Filtrado de señal

In [ ]:
def bandpass_filter(signal, fs=FS, low=BANDPASS[0], high=BANDPASS[1], order=4):
    sos = butter(order, [low, high], btype='bandpass', fs=fs, output='sos')
    return sosfiltfilt(sos, signal)

def notch_filter(signal, fs=FS, freq=NOTCH_HZ, Q=30):
    b, a = iirnotch(freq, Q, fs)
    return sosfiltfilt(butter(1, [1,60], btype='bandpass', fs=fs, output='sos'), signal)

def preprocess_signal(signal, fs=FS):
    """Bandpass 1-45 Hz + notch 50 Hz."""
    # Bandpass
    sos_bp  = butter(4, BANDPASS, btype='bandpass', fs=fs, output='sos')
    sig_bp  = sosfiltfilt(sos_bp, signal)
    # Notch 50 Hz
    b, a    = iirnotch(NOTCH_HZ, Q=30, fs=fs)
    sig_out = sp_signal.filtfilt(b, a, sig_bp)
    return sig_out

# ── Demo visual ──────────────────────────────────────────────────────────────
import pandas as _pd
_df    = _pd.read_csv(glob.glob(os.path.join(DATA_DIR,'P04.txt'))[0],
                      comment='%', skipinitialspace=True)
_raw   = _df['EXG Channel 0'].values[:FS*10].astype(float)
_filt  = preprocess_signal(_raw)
_t     = np.arange(len(_raw)) / FS

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(_t, _raw,  lw=0.6, color='gray');  axes[0].set_title('P04 – Fp1 cruda');     axes[0].set_ylabel('µV')
axes[1].plot(_t, _filt, lw=0.6, color='steelblue'); axes[1].set_title('P04 – Fp1 filtrada'); axes[1].set_ylabel('µV'); axes[1].set_xlabel('s')
plt.tight_layout(); plt.show()

---
## 4 · Epoching (ventana 250 muestras, paso 125)

In [ ]:
def make_epochs(signals_dict, epoch_len=EPOCH_SAMPLES, step=STEP_SAMPLES):
    """
    signals_dict : {ch_name: np.array(n_samples)}
    Retorna lista de dicts {ch_name: array(epoch_len)} con el índice de inicio.
    """
    n = next(iter(signals_dict.values())).shape[0]
    epochs = []
    starts = range(0, n - epoch_len + 1, step)
    for s in starts:
        epoch = {ch: sig[s:s+epoch_len] for ch, sig in signals_dict.items()}
        epoch['_start_sample'] = s
        epochs.append(epoch)
    return epochs

print(f'Epoching: ventana={EPOCH_SAMPLES} muestras ({EPOCH_SAMPLES/FS:.1f} s), '
      f'paso={STEP_SAMPLES} muestras ({STEP_SAMPLES/FS:.1f} s), '
      f'solape={100*(1-STEP_SAMPLES/EPOCH_SAMPLES):.0f} %')

---
## 5 · Rechazo de épocas malas (±150 µV)

In [ ]:
def reject_epoch(epoch, threshold=AMP_THRESH):
    """
    Retorna True si la época debe descartarse.
    Criterio: algún canal supera ±threshold µV en cualquier muestra.
    """
    for ch, sig in epoch.items():
        if ch.startswith('_'):
            continue
        if np.any(np.abs(sig) > threshold):
            return True
    return False

def filter_epochs(epochs, threshold=AMP_THRESH):
    kept    = [e for e in epochs if not reject_epoch(e, threshold)]
    dropped = len(epochs) - len(kept)
    return kept, dropped

print(f'Umbral de rechazo: ±{AMP_THRESH} µV')

---
## 6 · Extracción de 273 features por época

| Bloque | Fórmula | N |
|--------|---------|---|
| `abs_<banda>_<ch>` | potencia absoluta via Welch | 80 |
| `rel_<banda>_<ch>` | potencia absoluta / potencia total | 80 |
| `ratio_<r>_<ch>` | 5 ratios estándar por canal | 80 |
| `asym_<banda>_<par>` | ln(P_der) − ln(P_izq) | 32 |
| `broadband_total` | suma de todas las bandas, todos los canales | 1 |

In [ ]:
def band_power(sig, fs, lo, hi, nperseg=None):
    """Potencia integrada en [lo, hi] Hz usando Welch."""
    if nperseg is None:
        nperseg = min(len(sig), fs * 2)
    freqs, psd = sp_signal.welch(sig, fs=fs, nperseg=nperseg)
    mask = (freqs >= lo) & (freqs <= hi)
    return float(np.trapz(psd[mask], freqs[mask]))


def extract_features(epoch, valid_chs, ch_names=CH_NAMES, fs=FS):
    """
    epoch      : dict {ch_name: np.array(250)}
    valid_chs  : lista de canales a usar (excluye saturados)
    Retorna    : dict con 273 features.
    """
    feats = {}

    # ── 1. Potencias por canal × banda ──────────────────────────────────────
    pow_abs = {}   # {ch: {banda: valor}}
    for ch in ch_names:
        if ch not in valid_chs or ch not in epoch:
            pow_abs[ch] = {b: np.nan for b in BANDS}
            continue
        sig = epoch[ch].astype(float)
        bp  = {b: band_power(sig, fs, lo, hi) for b, (lo, hi) in BANDS.items()}
        pow_abs[ch] = bp

    # ── 2. Potencia absoluta (80) ────────────────────────────────────────────
    for ch in ch_names:
        for b in BANDS:
            feats[f'abs_{b}_{ch}'] = pow_abs[ch][b]

    # ── 3. Potencia relativa (80) ────────────────────────────────────────────
    for ch in ch_names:
        total_ch = sum(v for v in pow_abs[ch].values() if not np.isnan(v))
        total_ch = total_ch if total_ch > 0 else np.nan
        for b in BANDS:
            feats[f'rel_{b}_{ch}'] = pow_abs[ch][b] / total_ch

    # ── 4. Ratios espectrales (80) ───────────────────────────────────────────
    # 5 ratios × 16 canales
    _eps = 1e-10
    for ch in ch_names:
        d = {b: pow_abs[ch][b] for b in BANDS}
        feats[f'ratio_theta_alpha_{ch}']         = d['theta'] / (d['alpha'] + _eps)
        feats[f'ratio_beta_alpha_{ch}']           = d['beta']  / (d['alpha'] + _eps)
        feats[f'ratio_theta_beta_{ch}']           = d['theta'] / (d['beta']  + _eps)
        feats[f'ratio_delta_alpha_{ch}']          = d['delta'] / (d['alpha'] + _eps)
        # Engagement index = beta / (alpha + theta)  [Pope et al.]
        feats[f'ratio_engagement_{ch}']           = d['beta']  / (d['alpha'] + d['theta'] + _eps)

    # ── 5. Asimetría hemisférica (32) ────────────────────────────────────────
    # ln(P_der) − ln(P_izq) para 8 pares × 4 bandas (theta, alpha, beta, gamma)
    for (left_ch, right_ch) in ASYM_PAIRS:
        for b in ASYM_BANDS:
            p_l = pow_abs.get(left_ch,  {}).get(b, np.nan)
            p_r = pow_abs.get(right_ch, {}).get(b, np.nan)
            if p_l is not None and p_r is not None and p_l > 0 and p_r > 0:
                val = np.log(p_r) - np.log(p_l)
            else:
                val = np.nan
            pair_name = f'{left_ch}_{right_ch}'
            feats[f'asym_{b}_{pair_name}'] = val

    # ── 6. Potencia broadband total (1) ─────────────────────────────────────
    all_vals = [pow_abs[ch][b] for ch in ch_names for b in BANDS
                if not np.isnan(pow_abs[ch][b])]
    feats['broadband_total'] = float(np.sum(all_vals)) if all_vals else np.nan

    return feats


# Verificar conteo de features con un epoch de prueba
import pandas as _pd2
_df2     = _pd2.read_csv(glob.glob(os.path.join(DATA_DIR,'P04.txt'))[0],
                         comment='%', skipinitialspace=True)
_sig_dict = {ch: preprocess_signal(_df2[COL_MAP[ch]].values[:EPOCH_SAMPLES].astype(float))
             for ch in CH_NAMES}
_test_epoch = {**_sig_dict, '_start_sample': 0}
_test_feats = extract_features(_test_epoch, valid_chs=CH_NAMES)
print(f'Features extraídas: {len(_test_feats)}  (objetivo: 273)')
assert len(_test_feats) == 273, f'ERROR: se esperaban 273, se obtuvieron {len(_test_feats)}'

---
## 7 · Etiquetado de estímulos / páginas

In [ ]:
# ── Opción B: carga desde CSV ─────────────────────────────────────────────────
if USE_EVENTS_FILE:
    events_df = pd.read_csv(EVENTS_CSV)
    # Formato esperado: participant | page | start_sample | end_sample
    print(f'Eventos cargados: {len(events_df)} filas')
    print(events_df.head())

# ── Opción A: bloques fijos ───────────────────────────────────────────────────
PAGE_LEN_SAMPLES   = int(PAGE_DURATION_S   * FS)
START_OFFSET_SAMPLES = int(PAGE_START_OFFSET_S * FS)

def get_page_label(start_sample, participant_id=None):
    """
    Retorna el número de página (1-based) para un epoch que comienza en start_sample.
    Si el epoch cae fuera del protocolo → None (se descarta).
    """
    if USE_EVENTS_FILE and participant_id is not None:
        pid_events = events_df[events_df['participant'] == participant_id]
        match = pid_events[
            (pid_events['start_sample'] <= start_sample) &
            (pid_events['end_sample']   >  start_sample)
        ]
        return int(match['page'].values[0]) if len(match) == 1 else None

    # Opción A — bloques fijos
    adjusted = start_sample - START_OFFSET_SAMPLES
    if adjusted < 0:
        return None   # antes del inicio del protocolo
    page = adjusted // PAGE_LEN_SAMPLES + 1
    if page > N_PAGES:
        return None   # fuera del protocolo
    return int(page)

# Demo
for s in [0, PAGE_LEN_SAMPLES//2, PAGE_LEN_SAMPLES, PAGE_LEN_SAMPLES*2, PAGE_LEN_SAMPLES*N_PAGES+1]:
    print(f'  sample {s:6d}  →  página {get_page_label(s)}')

---
## 8 · Pipeline completo — todos los participantes

In [ ]:
all_records   = []   # lista de dicts (una fila por época)
rejection_log = []   # log de rechazo por participante

files = sorted(glob.glob(os.path.join(DATA_DIR, 'P*.txt')))
print(f'Procesando {len(files)} participantes…\n')

for fpath in files:
    pid_str = os.path.basename(fpath).replace('.txt', '')
    valid   = valid_channels(pid_str)

    # 1. Cargar
    df_raw = pd.read_csv(fpath, comment='%', skipinitialspace=True)
    df_raw.columns = df_raw.columns.str.strip()

    # 2. Filtrar canales válidos
    sigs = {}
    for ch in valid:
        col = COL_MAP[ch]
        if col in df_raw.columns:
            sigs[ch] = preprocess_signal(df_raw[col].values.astype(float))

    if not sigs:
        print(f'  {pid_str}: sin canales válidos — omitido')
        continue

    # 3. Epoching
    epochs = make_epochs(sigs)
    n_total = len(epochs)

    # 4. Rechazo
    epochs_ok, n_dropped = filter_epochs(epochs)
    pct_kept = 100 * len(epochs_ok) / n_total if n_total else 0

    # 5. Features + etiquetado
    n_labeled = 0
    for ep in epochs_ok:
        start  = ep['_start_sample']
        page   = get_page_label(start, participant_id=pid_str)
        if page is None:
            continue

        feats = extract_features(ep, valid_chs=valid)
        feats['participant']   = pid_str
        feats['page']          = page
        feats['start_sample']  = start
        feats['start_time_s']  = start / FS
        feats['n_valid_chs']   = len(valid)
        all_records.append(feats)
        n_labeled += 1

    rejection_log.append({
        'participant'  : pid_str,
        'n_epochs_raw' : n_total,
        'n_dropped_amp': n_dropped,
        'n_outside_prot': len(epochs_ok) - n_labeled,
        'n_kept'       : n_labeled,
        'pct_kept'     : round(pct_kept, 1),
        'n_valid_chs'  : len(valid),
    })
    print(f'  {pid_str}: {n_total} épocas → {n_dropped} rechazadas ({100*n_dropped/n_total:.1f}%) '
          f'→ {n_labeled} etiquetadas  [{len(valid)} ch válidos]')

print(f'\nTotal de épocas finales: {len(all_records)}')

---
## 9 · Resumen de rechazo por participante

In [ ]:
log_df = pd.DataFrame(rejection_log)
print(log_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(log_df['participant'], log_df['pct_kept'], color='steelblue')
axes[0].axhline(log_df['pct_kept'].median(), color='k', ls='--', lw=1)
axes[0].set_title('% épocas conservadas'); axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=90); axes[0].set_ylim(0, 105)

axes[1].bar(log_df['participant'], log_df['n_kept'], color='seagreen')
axes[1].set_title('N° épocas finales por participante')
axes[1].tick_params(axis='x', rotation=90)

axes[2].bar(log_df['participant'], log_df['n_valid_chs'], color='tomato')
axes[2].set_title('N° canales válidos por participante')
axes[2].tick_params(axis='x', rotation=90)

plt.tight_layout(); plt.show()

print(f'\nMediana épocas conservadas: {log_df["pct_kept"].median():.1f} %')
print(f'Participantes con < 50 % épocas: {(log_df["pct_kept"] < 50).sum()}')

---
## 10 · Dataset final y guardado

In [ ]:
epochs_df = pd.DataFrame(all_records)

# Ordenar columnas: metadatos primero, features después
meta_cols  = ['participant','page','start_sample','start_time_s','n_valid_chs']
feat_cols  = [c for c in epochs_df.columns if c not in meta_cols]
epochs_df  = epochs_df[meta_cols + feat_cols]

print(f'Shape final: {epochs_df.shape}')
print(f'Participantes: {epochs_df["participant"].nunique()}')
print(f'Páginas únicas: {sorted(epochs_df["page"].unique())}')
print(f'Features: {len(feat_cols)}')
print(f'NaN (%) por columna (top 10):')
print((epochs_df[feat_cols].isna().mean() * 100).sort_values(ascending=False).head(10).round(2))

epochs_df.head(3)

In [ ]:
# Distribución de épocas por participante y página
pivot = epochs_df.groupby(['participant','page']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns)*0.8), max(5, len(pivot)*0.3)))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGn')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, fontsize=8)
ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index,   fontsize=7)
ax.set_xlabel('Página'); ax.set_ylabel('Participante')
ax.set_title('N° épocas por participante y página')
plt.colorbar(im, ax=ax, label='N° épocas')
plt.tight_layout(); plt.show()

In [ ]:
# Guardar dataset completo
out_path = os.path.join(OUT_DIR, 'epochs_features.parquet')
epochs_df.to_parquet(out_path, index=False)
print(f'Guardado: {out_path}')

# También CSV (más pesado pero portable)
csv_path = os.path.join(OUT_DIR, 'epochs_features.csv')
epochs_df.to_csv(csv_path, index=False)
print(f'Guardado: {csv_path}')

# Log de rechazo
log_path = os.path.join(OUT_DIR, 'rejection_log.csv')
log_df.to_csv(log_path, index=False)
print(f'Guardado: {log_path}')

print(f'\nDataset final: {epochs_df.shape[0]} épocas × {epochs_df.shape[1]} columnas')

---
## 11 · Inspección rápida de features

In [ ]:
# Potencia alpha media por participante
alpha_cols = [c for c in feat_cols if c.startswith('abs_alpha_')]
epochs_df['alpha_mean'] = epochs_df[alpha_cols].mean(axis=1)

fig, ax = plt.subplots(figsize=(14, 4))
mu  = epochs_df.groupby('participant')['alpha_mean'].median()
mu.plot.bar(ax=ax, color='steelblue')
ax.set_title('Potencia alpha mediana por participante (todas las épocas)')
ax.set_ylabel('Potencia alpha absoluta (µV²)')
ax.tick_params(axis='x', rotation=90)
plt.tight_layout(); plt.show()

In [ ]:
# Asimetría alpha frontal (FAA) — F3/F4
faa_col = 'asym_alpha_F3_F4'
if faa_col in epochs_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].boxplot(
        [epochs_df[epochs_df['participant']==p][faa_col].dropna().values
         for p in epochs_df['participant'].unique()],
        tick_labels=epochs_df['participant'].unique(), showfliers=False
    )
    axes[0].axhline(0, color='k', ls='--', lw=0.8)
    axes[0].set_title('FAA (F3/F4 alpha asymmetry) por participante')
    axes[0].set_ylabel('ln(P_F4) − ln(P_F3)')
    axes[0].tick_params(axis='x', rotation=90)

    epochs_df.groupby('page')[faa_col].median().plot.bar(ax=axes[1], color='coral')
    axes[1].axhline(0, color='k', ls='--', lw=0.8)
    axes[1].set_title('FAA mediana por página')
    axes[1].set_ylabel('ln(P_F4) − ln(P_F3)')
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout(); plt.show()

In [ ]:
# Perfil de bandas promedio — todos los participantes
band_means = {}
for b in BANDS:
    cols = [c for c in feat_cols if c.startswith(f'abs_{b}_')]
    band_means[b] = epochs_df[cols].values.flatten()

fig, ax = plt.subplots(figsize=(10, 4))
positions = range(len(BANDS))
bp = ax.boxplot([band_means[b] for b in BANDS], tick_labels=list(BANDS.keys()),
                showfliers=False, patch_artist=True)
colors_b = ['#4472C4','#ED7D31','#A9D18E','#FF0000','#7030A0']
for patch, c in zip(bp['boxes'], colors_b):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_yscale('log')
ax.set_title('Distribución de potencia absoluta por banda (todas las épocas, todos los participantes)')
ax.set_ylabel('Potencia absoluta (µV²) — escala log')
plt.tight_layout(); plt.show()